> 在线讲义：[第 15 章 · OpenVLA-OFT：一次吐完所有动作，直接回归连续值](https://wikieden.github.io/vla-from-scratch/lessons/15-oft.html) · 本书《VLA 从零到 π0》

# 第 15 章 · OpenVLA-OFT：一次吐完所有动作，直接回归连续值

配套讲义：仓库里 `vla-deep-dive/notes/lessons/15-oft.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `throughput(k, latency)`、`TABLE_II` | 4.2 / 15.9 / 108.8 / 109.7 / 71.4 Hz；26 倍 |
| 第 2 步 | `l2_fit(values, weights)`、`l1_fit(values, weights)` | 60/40 时 L2 = +0.01、L1 = +0.05 |
| 第 2 步 | `DIFFUSION_STEPS` | 50 步 4.2 Hz、1 步 109.4 Hz |
| 第 3 步 | `film(f, gamma, beta)` | [0.6, −0.1, 0.5] |

数据取自 openvla-oft-2502.19645.txt 表 I、表 II（A100，一张 224 px 图 + LIBERO 指令）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

In [ ]:
TABLE_II = [
    ("OpenVLA（逐词念）", 1, 0.2396),
    ("+ PD", 1, 0.0629),
    ("+ PD&AC", 8, 0.0735),
    ("+ PD&AC, Cont-L1", 8, 0.0729),
    ("+ 腕部相机 + 本体状态", 8, 0.1120),
]
DIFFUSION_STEPS = {50: (1.9070, 91.1), 10: (0.4145, 91.0), 5: (0.2279, 90.0), 2: (0.0996, 85.7), 1: (0.0731, 0.0)}
TABLE_I = {"OpenVLA 微调": 76.5, "+ PD&AC": 90.2, "+ PD&AC + 扩散": 95.4, "OFT（L1）": 95.3, "OFT + 腕部 + 状态": 97.1}

**函数 `throughput`**：每次前向出 k 个动作、花 latency 秒，一秒能出几个动作。

In [ ]:
def throughput(k, latency):
    """每次前向出 k 个动作、花 latency 秒，一秒能出几个动作。"""
    return k / latency

**函数 `l2_fit`**：L2 损失的最优常数解：加权平均。

In [ ]:
def l2_fit(values, weights):
    """L2 损失的最优常数解：加权平均。"""
    v, w = np.asarray(values, float), np.asarray(weights, float)
    return float((v * w).sum() / w.sum())

**函数 `l1_fit`**：L1 损失的最优常数解：加权中位数（在候选点上直接比总误差）。

In [ ]:
def l1_fit(values, weights):
    """L1 损失的最优常数解：加权中位数（在候选点上直接比总误差）。"""
    v, w = np.asarray(values, float), np.asarray(weights, float)
    costs = [float((w * np.abs(v - c)).sum()) for c in v]
    return float(v[int(np.argmin(costs))])

**函数 `film`**：FiLM：(1 + γ) ⊙ F + β，γ、β 对所有图块共享。

In [ ]:
def film(f, gamma, beta):
    """FiLM：(1 + γ) ⊙ F + β，γ、β 对所有图块共享。"""
    return (1 + np.asarray(gamma, float)) * np.asarray(f, float) + np.asarray(beta, float)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 并行解码：吞吐 = K / 延迟")
for name, k, t in TABLE_II:
    print(f"  {name:<16} K = {k}  延迟 {t:.4f} s  吞吐 {throughput(k, t):6.1f} Hz")
speedup = throughput(8, 0.0735) / throughput(1, 0.2396)
print(f"  PD&AC 比原版快 {speedup:.1f} 倍（论文写 26×）")

In [ ]:
print("\n第二步 L1 取中位数，L2 取平均")
for split in ((0.5, 0.5), (0.6, 0.4)):
    vals, w = [0.05, -0.05], list(split)
    print(f"  左 {split[0]:.0%} / 右 {split[1]:.0%}：L2 → {l2_fit(vals, w):+.3f}，L1 → {l1_fit(vals, w):+.3f}")
for name, sr in TABLE_I.items():
    print(f"  {name:<14} {sr:.1f}%")
for steps, (t, sr) in DIFFUSION_STEPS.items():
    print(f"  扩散 {steps:>2} 步：延迟 {t:.4f} s、吞吐 {throughput(8, t):6.1f} Hz、LIBERO-Long {sr:.1f}%")

In [ ]:
print("\n第三步 FiLM")
out = film([0.5, -0.2, 1.0], [0.2, 0, -0.5], [0, 0.1, 0])
print(f"  (1 + γ) ⊙ F + β = {np.round(out, 3).tolist()}")
print(f"  γ = β = 0 时不变：{film([0.5, -0.2, 1.0], [0, 0, 0], [0, 0, 0]).tolist()}")

## 3. 代码里才看得到的

1. `l1_fit` 在候选点上直接比较总误差；五五开时左右两个峰误差一样大，代码返回先找到的那个，这正是"L1 在五五开时不稳定"。
2. 表 II 的吞吐四舍五入到 0.1 Hz：1 / 0.2396 = 4.17，表里写 4.2；8 / 1.9070 = 4.195，也写 4.2。
3. `film` 的 γ、β 对所有图块共享，所以输入可以是 (图块数, D) 的矩阵，广播一次就行。

## 4. 自己改一改

- 把 `throughput` 的 K 改成 4、16、32，看延迟不变时吞吐怎么涨。
- 给 `l1_fit` 喂一个三峰分布（30% / 30% / 40%），看它站到哪里。
- 让 `film` 的 γ 取 −1，看对应通道变成了什么。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 念号码一位一位，填空格一次填满，再一次填八步；
> 平方取平均站山谷，绝对值取中位站多数那座峰；
> 戴眼镜按指令调色，一开始眼镜是透明的。